# 🧪 BlogMS — Auth Tests (`test_auth.py`)

| # | Test | الوصف |
|---|------|-------|
| 1 | `test_register_success` | تسجيل يوزر جديد بنجاح |
| 2 | `test_register_duplicate_username` | رفض username مكرر |
| 3 | `test_register_duplicate_email` | رفض email مكرر |
| 4 | `test_login_success` | تسجيل دخول ناجح |
| 5 | `test_login_wrong_password` | رفض كلمة مرور خاطئة |
| 6 | `test_login_nonexistent_user` | رفض يوزر غير موجود |
| 7 | `test_get_my_profile` | جلب بيانات اليوزر الحالي |
| 8 | `test_get_my_profile_unauthorized` | رفض الطلب بدون توكن |
| 9 | `test_invalid_token` | رفض توكن مزيف |

## 📝 `test_register_success`

In [ ]:
def test_register_success(client):
    """✅ 201 · username صح · role=reader · hashed_password مش بيتبعت"""
    resp = client.post("/auth/register", json={
        "username": "newuser", "email": "new@test.com", "password": "pass123"
    })
    assert resp.status_code == 201
    data = resp.json()
    assert data["username"] == "newuser"
    assert data["role"] == "reader"
    assert "hashed_password" not in data

## 📝 `test_register_duplicate_username`

In [ ]:
def test_register_duplicate_username(client):
    """❌ 400 · Username already registered"""
    payload = {"username": "dupuser", "email": "dup@test.com", "password": "pass123"}
    client.post("/auth/register", json=payload)
    resp = client.post("/auth/register", json={**payload, "email": "other@test.com"})
    assert resp.status_code == 400
    assert "Username already registered" in resp.json()["detail"]

## 📝 `test_register_duplicate_email`

In [ ]:
def test_register_duplicate_email(client):
    """❌ 400 · Email already registered"""
    client.post("/auth/register", json={"username": "user1", "email": "same@test.com", "password": "pass123"})
    resp = client.post("/auth/register", json={"username": "user2", "email": "same@test.com", "password": "pass123"})
    assert resp.status_code == 400
    assert "Email already registered" in resp.json()["detail"]

## 📝 `test_login_success`

In [ ]:
def test_login_success(client):
    """✅ 200 · access_token موجود · token_type=bearer"""
    client.post("/auth/register", json={"username": "loginuser", "email": "login@test.com", "password": "mypass123"})
    resp = client.post("/auth/login", data={"username": "loginuser", "password": "mypass123"})
    assert resp.status_code == 200
    assert "access_token" in resp.json()
    assert resp.json()["token_type"] == "bearer"

## 📝 `test_login_wrong_password`

In [ ]:
def test_login_wrong_password(client):
    """❌ 401 · كلمة مرور خاطئة"""
    client.post("/auth/register", json={"username": "wrongpass", "email": "wp@test.com", "password": "correct"})
    resp = client.post("/auth/login", data={"username": "wrongpass", "password": "wrong"})
    assert resp.status_code == 401

## 📝 `test_login_nonexistent_user`

In [ ]:
def test_login_nonexistent_user(client):
    """❌ 401 · يوزر غير موجود"""
    resp = client.post("/auth/login", data={"username": "ghost", "password": "pass"})
    assert resp.status_code == 401

## 📝 `test_get_my_profile`

In [ ]:
def test_get_my_profile(client, reader_token):
    """✅ 200 · username=readeruser"""
    resp = client.get("/users/me", headers={"Authorization": f"Bearer {reader_token}"})
    assert resp.status_code == 200
    assert resp.json()["username"] == "readeruser"

## 📝 `test_get_my_profile_unauthorized`

In [ ]:
def test_get_my_profile_unauthorized(client):
    """❌ 401 · بدون توكن"""
    resp = client.get("/users/me")
    assert resp.status_code == 401

## 📝 `test_invalid_token`

In [ ]:
def test_invalid_token(client):
    """❌ 401 · توكن مزيف"""
    resp = client.get("/users/me", headers={"Authorization": "Bearer invalidtoken"})
    assert resp.status_code == 401

## ▶️ تشغيل الـ Tests

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_auth.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print("STDERR:\n", result.stderr)